<a href="https://colab.research.google.com/github/elsheppardo/hello-world/blob/main/Stage_4_Capital_Gains_Events.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
STAGE 4: 2022 Capital Gains Events
=====================================
Purpose: Calculate the realized CAD capital gain/(loss) on every
         2022 disposition event, using:
           - CAD valuations from Stage 1 (FX rates)
           - USD ACB from Stage 2 (for USD dispositions — N/A in 2022)
           - BTC ACB from Stage 3 (for BTC dispositions)

Output: Stage4_2022_Events.xlsx

TAX TREATMENT — TERRA / ANCHOR
==============================
aUST (Anchor Protocol receipt token) is a value-accruing receipt
token, mechanically similar to wstETH and sAVAX. The token count in
the wallet does not grow during holding — only the exchange ratio
between aUST and underlying UST drifts upward.

Under CRA's commodity-based treatment of crypto, the redemption of a
value-accruing receipt token is a single capital disposition event.
This approach:
  - UST deposited into Anchor and aUST received at the same FMV
    → no gain/loss on deposit (practical zero, since UST was $1.00).
  - aUST held while exchange ratio drifts → unrealized appreciation,
    no taxable event during holding.
  - aUST redeemed for UST on May 10, 2022 → single taxable disposition.
  - Proceeds of aUST disposition = FMV of UST received × FX.
  - Because UST depegged BEFORE redemption, proceeds were valued at
    ~$0.90/UST not $1.00.
  - ACB of aUST = CAD value of UST originally deposited.
  - The ~10,224 UST "yield" that appeared on redemption is subsumed
    into the aUST disposition calculation (capital gains treatment),
    not reported as separate interest income.

Major events covered:
  A. Avalanche → Terra bridge (Feb 24, 2022)
     - AVAX/LUNA disposal at $33,950 CAD carry-forward ACB
     - UST acquired and deposited to Anchor (non-event chain, same FMV)

  B. Paxos BTC Buys (March 2022)
     - Three aggregated buy days (Mar 5, Mar 10, Mar 14)

  C. BTC → UST via KuCoin (March 2022)
     - Large aggregated BTC disposition (7.31 BTC → UST)
     - UST acquired then immediately deposited to Anchor

  D. May 10, 2022 — Terra Collapse (MAJOR)
     - SINGLE aUST disposition event
     - Proceeds = total UST received × $0.90 × FX
     - ACB = total CAD value of all UST originally deposited

  E. BTC Round-Trips (Jun-Aug 2022)
     - Two Paxos sells (Jul 1, Jul 14) — uses BTC ACB
     - Three KuCoin/Paxos buys — acquisitions only

Key output: Cell K of the 2022 subtotal row — net CAD capital gain/(loss)
"""

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

FONT_NAME = "Arial"
styles = {
    'title':       Font(name=FONT_NAME, size=16, bold=True, color='1F4E79'),
    'section':     Font(name=FONT_NAME, size=12, bold=True, color='1F4E79'),
    'header':      Font(name=FONT_NAME, size=10, bold=True, color='FFFFFF'),
    'body':        Font(name=FONT_NAME, size=10),
    'body_bold':   Font(name=FONT_NAME, size=10, bold=True),
    'input':       Font(name=FONT_NAME, size=10, color='0000FF'),
    'formula':     Font(name=FONT_NAME, size=10, color='000000'),
    'link':        Font(name=FONT_NAME, size=10, color='008000'),
    'note':        Font(name=FONT_NAME, size=9,  italic=True, color='595959'),
    'header_fill':    PatternFill('solid', fgColor='1F4E79'),
    'subheader_fill': PatternFill('solid', fgColor='D5E8F0'),
    'verify_fill':    PatternFill('solid', fgColor='FFF2CC'),
    'total_fill':     PatternFill('solid', fgColor='E2EFDA'),
    'warning_fill':   PatternFill('solid', fgColor='FCE4D6'),
    'acq_fill':       PatternFill('solid', fgColor='F2F2F2'),  # light gray for acquisitions
    'disp_fill':      PatternFill('solid', fgColor='FFF7E6'),  # light yellow for dispositions
}
thin = Side(border_style='thin', color='BFBFBF')
cell_border = Border(left=thin, right=thin, top=thin, bottom=thin)

FMT_USD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_CAD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_CRYPTO = '#,##0.00000000'
FMT_PRICE  = '"$"#,##0.00'
FMT_FX     = '0.0000'


def build_workbook(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "2022_Events"

    # Column widths
    widths = [12, 14, 30, 12, 14, 14, 14, 10, 14, 14, 14, 50]
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Title
    ws.cell(row=1, column=1, value="2022 Capital Gains Events").font = styles['title']
    ws.cell(row=2, column=1, value=(
        "Acquisition rows: '-' in cols J/K. Disposition rows: compute gain/(loss) in col K. "
        "Terra/Anchor activity treated as capital gains (aUST as value-accruing receipt token)."
    )).font = styles['note']
    ws.cell(row=3, column=1, value=(
        "NOTE: This is a standalone workbook. FX rates and ACB values are hardcoded here "
        "(from Stages 1, 2, 3) for clarity — adjust at your accountant's direction."
    )).font = styles['note']

    # Headers
    headers = [
        "Date",               # A
        "Platform",           # B
        "Event",              # C
        "Asset",              # D
        "Amount",             # E
        "Price (USD)",        # F
        "USD Value",          # G
        "FX Rate",            # H
        "CAD Value",          # I
        "CAD ACB",            # J
        "CAD Gain/(Loss)",    # K
        "Notes",              # L
    ]
    for i, h in enumerate(headers, 1):
        cell = ws.cell(row=5, column=i, value=h)
        cell.font = styles['header']
        cell.fill = styles['header_fill']
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = cell_border

    # Helper to write a section header
    def write_section(r, title):
        c = ws.cell(row=r, column=1, value=title)
        c.font = styles['section']
        c.fill = styles['subheader_fill']
        for col in range(1, 13):
            ws.cell(row=r, column=col).fill = styles['subheader_fill']
            ws.cell(row=r, column=col).border = cell_border

    # Helper: write an acquisition row (no gain/loss)
    def write_acquisition(r, date, platform, event, asset, amount, price_usd, fx_rate, notes,
                           usd_override=None):
        ws.cell(row=r, column=1, value=date).font = styles['body']
        ws.cell(row=r, column=2, value=platform).font = styles['body']
        ws.cell(row=r, column=3, value=event).font = styles['body']
        ws.cell(row=r, column=4, value=asset).font = styles['body']
        c = ws.cell(row=r, column=5, value=amount); c.font = styles['input']; c.number_format = FMT_CRYPTO
        c = ws.cell(row=r, column=6, value=price_usd); c.font = styles['input']; c.number_format = FMT_PRICE
        if usd_override is not None:
            c = ws.cell(row=r, column=7, value=usd_override); c.font = styles['input']
        else:
            c = ws.cell(row=r, column=7, value=f"=E{r}*F{r}"); c.font = styles['formula']
        c.number_format = FMT_USD
        c = ws.cell(row=r, column=8, value=fx_rate); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
        c = ws.cell(row=r, column=9, value=f"=G{r}*H{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
        ws.cell(row=r, column=10, value="-").font = styles['body']
        ws.cell(row=r, column=10).alignment = Alignment(horizontal='center')
        ws.cell(row=r, column=11, value="-").font = styles['body']
        ws.cell(row=r, column=11).alignment = Alignment(horizontal='center')
        c = ws.cell(row=r, column=12, value=notes); c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        for col in range(1, 13):
            ws.cell(row=r, column=col).fill = styles['acq_fill']
            ws.cell(row=r, column=col).border = cell_border

    # Helper: write a disposition row (with gain/loss calculation)
    def write_disposition(r, date, platform, event, asset, amount, price_usd, fx_rate,
                          acb_cad, notes, usd_override=None, flag_price=False, flag_acb=False):
        ws.cell(row=r, column=1, value=date).font = styles['body']
        ws.cell(row=r, column=2, value=platform).font = styles['body']
        ws.cell(row=r, column=3, value=event).font = styles['body']
        ws.cell(row=r, column=4, value=asset).font = styles['body']
        c = ws.cell(row=r, column=5, value=amount); c.font = styles['input']; c.number_format = FMT_CRYPTO
        c = ws.cell(row=r, column=6, value=price_usd); c.font = styles['input']
        if flag_price: c.fill = styles['verify_fill']
        c.number_format = FMT_PRICE
        if usd_override is not None:
            c = ws.cell(row=r, column=7, value=usd_override); c.font = styles['input']
        else:
            c = ws.cell(row=r, column=7, value=f"=E{r}*F{r}"); c.font = styles['formula']
        c.number_format = FMT_USD
        c = ws.cell(row=r, column=8, value=fx_rate); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
        c = ws.cell(row=r, column=9, value=f"=G{r}*H{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=10, value=acb_cad); c.font = styles['input']
        if flag_acb: c.fill = styles['verify_fill']
        c.number_format = FMT_CAD
        c = ws.cell(row=r, column=11, value=f"=I{r}-J{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=12, value=notes); c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        for col in range(1, 13):
            ws.cell(row=r, column=col).fill = styles['disp_fill']
            ws.cell(row=r, column=col).border = cell_border

    # ============================================================
    # Build rows
    # ============================================================
    r = 6

    # --- Section A: Avalanche → Terra Bridge ---
    write_section(r, "Section A: Avalanche → Terra Bridge (Feb 24, 2022)")
    r += 1

    # AVAX disposition at carry-forward ACB
    write_disposition(
        r,
        date="2022-02-24", platform="Avalanche/KC",
        event="Dispose AVAX+LUNA (bridge)", asset="AVAX/LUNA",
        amount=350.46, price_usd=76.00, fx_rate=1.2734,
        acb_cad=33950.00,
        notes="Disposal of Avalanche-held assets. ACB = $33,950 CAD carry-forward from 2021 summary. "
              "AVAX market price on Feb 24 ~$76 (VERIFY). Small gain/loss from movement between 2021 close and Feb 24.",
        flag_price=True, flag_acb=True
    )
    r += 1

    # UST acquired from AVAX swap (at KuCoin, sent to Terra, deposited to Anchor)
    write_acquisition(
        r,
        date="2022-02-24", platform="Terra",
        event="Acquire UST from AVAX swap", asset="UST→aUST",
        amount=26465.88, price_usd=1.00, fx_rate=1.2734,
        notes="UST received from KuCoin after AVAX swap. Acquired at ~$1.00. "
              "Deposited to Anchor same day; aUST received at same FMV (non-event). "
              "CAD cost basis = col I — carries forward as aUST ACB."
    )
    ust_acq_row_1 = r
    r += 1

    # UST acquired unexplained source
    write_acquisition(
        r,
        date="2022-02-24", platform="Terra",
        event="Acquire UST (unexplained source)", asset="UST→aUST",
        amount=13453.64, price_usd=1.00, fx_rate=1.2734,
        notes="FLAG: Source not identified. ACB assumed at $1.00 USD × FX. "
              "Deposited to Anchor, aUST received at same FMV (non-event). "
              "CAD cost basis = col I — carries forward as aUST ACB."
    )
    # Flag notes cell
    ws.cell(row=r, column=12).fill = styles['warning_fill']
    ust_acq_row_2 = r
    r += 2

    # --- Section B: Paxos BTC Buys (March 2022) ---
    write_section(r, "Section B: Paxos BTC Buys (March 2022)")
    r += 1

    write_acquisition(
        r,
        date="2022-03-05", platform="Paxos",
        event="Buy BTC (25 fills aggregated)", asset="BTC",
        amount=2.93303373, price_usd=39376.29, fx_rate=1.2621,
        notes="Buy with USD from Mar 4 wire deposit ($114,970 USD). Paxos CSV aggregate.",
        usd_override=115484.03
    )
    r += 1

    write_acquisition(
        r,
        date="2022-03-10", platform="Paxos",
        event="Buy BTC (15 fills aggregated)", asset="BTC",
        amount=2.94941317, price_usd=39679.14, fx_rate=1.2740,
        notes="Paxos CSV aggregate.",
        usd_override=117026.78
    )
    r += 1

    write_acquisition(
        r,
        date="2022-03-14", platform="Paxos",
        event="Buy BTC (14 fills aggregated)", asset="BTC",
        amount=1.41420188, price_usd=37776.45, fx_rate=1.2810,
        notes="Buy with USD from Mar 9 wire ($171,845 USD). Paxos CSV aggregate.",
        usd_override=53420.78
    )
    r += 2

    # --- Section C: BTC → UST via KuCoin ---
    write_section(r, "Section C: BTC → UST via KuCoin (March 2022) — BTC Disposition")
    r += 1

    # BTC disposition — uses BTC ACB from Stage 3
    # From Stage 3, after the 3 Paxos buys + Feb 24 small deposit: ACB per BTC = $49,784.04
    # BTC disposed: 7.31441595
    # ACB CAD = 7.31441595 × $49,784.04 = $364,141.16
    write_disposition(
        r,
        date="2022-03-15", platform="KuCoin",
        event="Sell BTC → UST (aggregate)", asset="BTC",
        amount=7.31441595, price_usd=40125.03, fx_rate=1.2745,
        acb_cad=364141.16,
        notes="Aggregate of 7 BTC→UST swaps (Mar 8-17) + 0.021 BTC Feb 24 deposit. "
              "Proceeds USD = UST sent to Terra. ACB from Stage 3 BTC ledger (row 9 ACB per BTC × 7.31441595).",
        usd_override=293496.54
    )
    # Override col F with computed implied price
    ws.cell(row=r, column=6).value = f"=G{r}/E{r}"
    ws.cell(row=r, column=6).font = styles['formula']
    r += 1

    # UST acquired from BTC swap
    write_acquisition(
        r,
        date="2022-03-15", platform="KuCoin/Terra",
        event="Acquire UST from BTC swap", asset="UST→aUST",
        amount=293496.54, price_usd=1.00, fx_rate=1.2745,
        notes="UST acquired in KuCoin swap, sent to Terra, deposited to Anchor. "
              "aUST received at same FMV (non-event). "
              "CAD cost basis = col I — carries forward as aUST ACB."
    )
    ust_acq_row_3 = r
    r += 2

    # --- Section D: Terra Collapse (MAJOR) — MODEL B treatment ---
    write_section(r, "Section D: May 10, 2022 — Anchor (aUST) Disposition (MAJOR)")
    r += 1

    # Explanatory note row
    c = ws.cell(row=r, column=1, value=(
        "Treatment: aUST (Anchor receipt token) disposed on May 10 at FMV of UST received. "
        "Proceeds = 343,639.92 UST × $0.90 (depegged fill) × 1.3029 FX. "
        "ACB = sum of CAD value of all UST originally deposited to Anchor (rows above). "
        "Yield is subsumed into capital gain/(loss) of the receipt token — not separate interest income."
    ))
    c.font = styles['note']
    c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=12)
    for col in range(1, 13):
        ws.cell(row=r, column=col).fill = styles['warning_fill']
        ws.cell(row=r, column=col).border = cell_border
    ws.row_dimensions[r].height = 45
    r += 1

    # aUST disposition — the big loss
    # Amount: 275,188.20 aUST (main May 10 redemption — see Terra CSV)
    # But for simplicity in this row we show it as: effective proceeds of 343,639.92 UST received
    # Price (USD): $0.90 per UST received (the UST→USDT fill price on KuCoin)
    # FX: May 10 rate 1.3029
    # ACB: sum of CAD value of all UST that was deposited to Anchor
    #      (rows ust_acq_row_1, ust_acq_row_2, ust_acq_row_3 — these represent UST acquired
    #      then immediately deposited to Anchor at same FMV)
    acb_aust = f"=I{ust_acq_row_1}+I{ust_acq_row_2}+I{ust_acq_row_3}"
    ws.cell(row=r, column=1, value="2022-05-10").font = styles['body']
    ws.cell(row=r, column=2, value="Anchor/KuCoin").font = styles['body']
    ws.cell(row=r, column=3, value="Dispose aUST → UST → USDT").font = styles['body']
    ws.cell(row=r, column=4, value="aUST").font = styles['body']
    # Amount: use UST received as proxy (the aUST token count was 275,188 but proceeds denominated in UST)
    c = ws.cell(row=r, column=5, value=343639.92); c.font = styles['input']; c.number_format = FMT_CRYPTO
    c.comment = Comment(
        "Amount shown = UST received on aUST redemption (343,639.92). "
        "Underlying aUST redeemed = 275,188.20 (from Terra CSV cw20Send). "
        "Using UST amount because proceeds valuation is done per UST at $0.90 fill.",
        "Claude"
    )
    c = ws.cell(row=r, column=6, value=0.90); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_PRICE
    c = ws.cell(row=r, column=7, value=f"=E{r}*F{r}"); c.font = styles['formula']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=8, value=1.3029); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
    c = ws.cell(row=r, column=9, value=f"=G{r}*H{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=10, value=acb_aust); c.font = styles['formula']; c.number_format = FMT_CAD
    c.comment = Comment(
        "ACB = sum of CAD value of all UST deposited to Anchor (the 3 UST acquisition rows above). "
        "Since UST→aUST deposit was at same FMV (~$1.00), UST acquisition ACB = aUST ACB.",
        "Claude"
    )
    c = ws.cell(row=r, column=11, value=f"=I{r}-J{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=12, value=(
        "MAJOR CAPITAL LOSS. Single disposition event. "
        "UST depegged fill of $0.90 is user-estimated — VERIFY range $0.80–$0.95 with accountant."
    ))
    c.font = styles['note']; c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
    for col in range(1, 13):
        ws.cell(row=r, column=col).fill = styles['disp_fill']
        ws.cell(row=r, column=col).border = cell_border
    aust_disposition_row = r
    r += 1

    # USDT acquired (from subsequent UST→USDT swap on KuCoin)
    ws.cell(row=r, column=1, value="2022-05-10").font = styles['body']
    ws.cell(row=r, column=2, value="KuCoin").font = styles['body']
    ws.cell(row=r, column=3, value="Acquire USDT (UST → USDT swap)").font = styles['body']
    ws.cell(row=r, column=4, value="USDT").font = styles['body']
    c = ws.cell(row=r, column=5, value=f"=G{aust_disposition_row}"); c.font = styles['formula']; c.number_format = FMT_CRYPTO
    c = ws.cell(row=r, column=6, value=1.00); c.font = styles['input']; c.number_format = FMT_PRICE
    c = ws.cell(row=r, column=7, value=f"=E{r}*F{r}"); c.font = styles['formula']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=8, value=1.3029); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
    c = ws.cell(row=r, column=9, value=f"=G{r}*H{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
    ws.cell(row=r, column=10, value="-").font = styles['body']; ws.cell(row=r, column=10).alignment = Alignment(horizontal='center')
    ws.cell(row=r, column=11, value="-").font = styles['body']; ws.cell(row=r, column=11).alignment = Alignment(horizontal='center')
    c = ws.cell(row=r, column=12, value="USDT received ≈ USD proceeds value of aUST disposition. Feeds into USD pool (used in Stage 5).")
    c.font = styles['note']; c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
    for col in range(1, 13):
        ws.cell(row=r, column=col).fill = styles['acq_fill']
        ws.cell(row=r, column=col).border = cell_border
    r += 2

    # --- Section E: BTC Round-Trips (Jun-Aug 2022) ---
    write_section(r, "Section E: BTC Round-Trips (Jun-Aug 2022)")
    r += 1

    # Jun 30 KuCoin buy
    write_acquisition(
        r,
        date="2022-06-30", platform="KuCoin",
        event="Buy BTC with USDT", asset="BTC",
        amount=0.98105529, price_usd=18857.23, fx_rate=1.2886,
        notes="Rebuy 0.98 BTC on KuCoin (from KuCoin Trade History, first 8 fills — not the duplicated rows).",
        usd_override=18499.97
    )
    r += 1

    # Jul 1 Paxos sell — uses BTC ACB from Stage 3
    # At this point (after Jun 30 buy), Stage 3 ACB per BTC = $24,395.39
    # BTC disposed: 0.98005529
    # ACB CAD = 0.98005529 × 24,395.39 = $23,908.84
    write_disposition(
        r,
        date="2022-07-01", platform="Paxos",
        event="Sell BTC", asset="BTC",
        amount=0.98005529, price_usd=20338.49, fx_rate=1.2855,
        acb_cad=23908.84,
        notes="Sell on Paxos. Proceeds from Paxos CSV. ACB from Stage 3 BTC ledger (ACB per BTC after Jun 30 KuCoin buy).",
        usd_override=19931.06
    )
    r += 1

    # Jul 12 KuCoin buy
    write_acquisition(
        r,
        date="2022-07-12", platform="KuCoin",
        event="Buy BTC with USDT", asset="BTC",
        amount=0.99879120, price_usd=19523.60, fx_rate=1.3017,
        notes="Rebuy ~1 BTC on KuCoin (first 8 fills, not duplicated).",
        usd_override=19499.00
    )
    r += 1

    # Jul 14 Paxos sell
    # After Jul 12 buy, Stage 3 ACB per BTC = $25,407.79
    # BTC disposed: 0.99829120
    # ACB CAD = 0.99829120 × 25,407.79 = $25,364.38
    write_disposition(
        r,
        date="2022-07-14", platform="Paxos",
        event="Sell BTC", asset="BTC",
        amount=0.99829120, price_usd=20689.29, fx_rate=1.3075,
        acb_cad=25364.38,
        notes="Sell on Paxos. ACB from Stage 3 ledger.",
        usd_override=20653.41
    )
    r += 1

    # Aug 17 KuCoin buy
    write_acquisition(
        r,
        date="2022-08-17", platform="KuCoin",
        event="Buy BTC with USDT", asset="BTC",
        amount=0.99996931, price_usd=23399.72, fx_rate=1.2860,
        notes="Rebuy ~1 BTC on KuCoin (first 5 fills, not duplicated).",
        usd_override=23398.70
    )
    r += 1

    # Aug 17 Paxos buy
    write_acquisition(
        r,
        date="2022-08-17", platform="Paxos",
        event="Buy BTC with USD", asset="BTC",
        amount=1.00000000, price_usd=23430.39, fx_rate=1.2860,
        notes="Buy on Paxos using cash proceeds from earlier sells.",
        usd_override=23430.39
    )
    r += 2

    # --- 2022 Subtotal ---
    last_event_row = r - 2
    ws.cell(row=r, column=1, value="2022 Capital Gains Subtotal").font = styles['body_bold']
    c = ws.cell(row=r, column=11, value=f"=SUM(K6:K{last_event_row})")
    c.font = styles['body_bold']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=12, value="Sum of all K col values. Only disposition rows contribute (acquisitions have '-' which is ignored by SUM).")
    c.font = styles['note']
    for col in range(1, 13):
        ws.cell(row=r, column=col).fill = styles['total_fill']
        ws.cell(row=r, column=col).border = cell_border
    subtotal_row = r
    r += 2

    # --- Treatment note ---
    c = ws.cell(row=r, column=1, value=(
        "TREATMENT NOTE: aUST (Anchor receipt token) is treated as a value-accruing receipt token. "
        "The Anchor yield (~10,224 UST received on redemption) is subsumed into the capital disposition calculation — "
        "not reported as separate interest income. Redemption is a single capital disposition event."
    ))
    c.font = styles['note']
    c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=12)
    for col in range(1, 13):
        ws.cell(row=r, column=col).fill = styles['warning_fill']
        ws.cell(row=r, column=col).border = cell_border
    ws.row_dimensions[r].height = 45
    r += 3

    # --- Summary / reconciliation ---
    ws.cell(row=r, column=1, value="Summary").font = styles['section']
    r += 1
    ws.cell(row=r, column=1, value="2022 Net Capital Gain/(Loss) in CAD:").font = styles['body_bold']
    c = ws.cell(row=r, column=5, value=f"=K{subtotal_row}")
    c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_CAD
    r += 1
    ws.cell(row=r, column=1, value="Taxable portion (50% inclusion rate):").font = styles['body']
    c = ws.cell(row=r, column=5, value=f"=K{subtotal_row}*0.5")
    c.font = styles['formula']; c.number_format = FMT_CAD
    ws.cell(row=r, column=6, value="(50% of net capital loss = allowable capital loss)").font = styles['note']

    ws.freeze_panes = "A6"
    wb.save(output_path)
    print(f"Saved: {output_path}")


if __name__ == "__main__":
    build_workbook("Stage4_2022_Events.xlsx")